<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks pages by a transparent review score using visibility, staleness, position/CTR opportunity, engagement opportunity, and content depth. It does **not** use `trend_direction` or `trend_pct`, because those are outcome-derived fields.

The reason codes are:

- `stale_visible_content` — old content still receives visible demand.
- `low_ctr_visible_page` — the page is visible but may under-capture clicks.
- `low_engagement_visible_page` — visible traffic has a low measured engagement signal.
- `thin_visible_content` — a visible page has limited content depth.
- `monitor_only` — no strong review signal was found.

The action is deliberately narrow: review title/meta and intent, review depth and refresh, manual refresh review, review experience and intent, or monitor.

In [2]:
from pathlib import Path
import os, urllib.request
DATA_NAME = "content_refresh_anonymized.csv"
HERE = Path.cwd().resolve()
data_path = next((p / "data" / "raw" / DATA_NAME for p in [HERE, *HERE.parents] if (p / "data" / "raw" / DATA_NAME).exists()), None)
if data_path is None:
    ROOT = HERE / "flyrank_notebook_workspace"
    data_path = ROOT / "data" / "raw" / DATA_NAME
    data_path.parent.mkdir(parents=True, exist_ok=True)
    if not data_path.exists():
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv", data_path)
        except Exception as exc:
            raise FileNotFoundError("Extract the internship ZIP and open this notebook from it, or provide the public starter CSV.") from exc
else:
    ROOT = data_path.parents[2]
os.chdir(ROOT)
OUT = ROOT / "work" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", data_path)
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

df = pd.read_csv(data_path)

# Transparent, pre-decision action score. It does not use trend_direction or trend_pct.
impressions = pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0).clip(lower=0)
staleness = pd.to_numeric(df["days_since_last_update"], errors="coerce").fillna(0)
position = pd.to_numeric(df["avg_position"], errors="coerce").fillna(999)
ctr = pd.to_numeric(df["ctr"], errors="coerce").fillna(999)
engagement = pd.to_numeric(df["engagement_rate"], errors="coerce").fillna(999)
words = pd.to_numeric(df["word_count"], errors="coerce").fillna(0)

visibility = np.log1p(impressions) / np.log1p(impressions).max()
stale_risk = (staleness >= 180).astype(float)
ctr_opportunity = ((impressions >= 500) & (position.between(1, 20)) & (ctr < 0.5)).astype(float)
engagement_opportunity = ((impressions >= 500) & (engagement < 30) & (engagement > 0)).astype(float)
thin_visible = ((impressions >= 250) & (words > 0) & (words < 1200)).astype(float)

review_score = 100 * (0.35 * visibility + 0.25 * stale_risk + 0.20 * ctr_opportunity + 0.10 * engagement_opportunity + 0.10 * thin_visible)

def reasons(i):
    result = []
    if stale_risk.iat[i]: result.append("stale_visible_content")
    if ctr_opportunity.iat[i]: result.append("low_ctr_visible_page")
    if engagement_opportunity.iat[i]: result.append("low_engagement_visible_page")
    if thin_visible.iat[i]: result.append("thin_visible_content")
    return " | ".join(result) if result else "monitor_only"

def action(reason_text):
    if "low_ctr_visible_page" in reason_text: return "review_title_meta_and_intent"
    if "thin_visible_content" in reason_text: return "review_depth_and_refresh"
    if "stale_visible_content" in reason_text: return "manual_refresh_review"
    if "low_engagement_visible_page" in reason_text: return "review_experience_and_intent"
    return "monitor"

queue = pd.DataFrame({
    "review_score": review_score.round(1),
    "impressions_90d": impressions.astype(int),
    "days_since_last_update": staleness.astype(int),
    "avg_position": position,
    "ctr": ctr,
    "engagement_rate": engagement,
})
queue["reason_codes"] = [reasons(i) for i in range(len(queue))]
queue["suggested_action"] = queue["reason_codes"].map(action)
queue["confidence_note"] = np.where(queue["reason_codes"].eq("monitor_only"), "No strong review signal; monitor only.", "Review after checking intent, seasonality, and technical context.")
queue = queue.sort_values(["review_score", "impressions_90d"], ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)
display(queue.head(20))


Dataset: /content/flyrank_notebook_workspace/data/raw/content_refresh_anonymized.csv


,rank,review_score,impressions_90d,days_since_last_update,avg_position,ctr,engagement_rate,reason_codes,suggested_action,confidence_note
0,1,84.3,61678,194,19.7,0.15,0.84,stale_visible_content | low_ctr_visible_page |...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
1,2,80.3,13299,193,10.5,0.49,5.13,stale_visible_content | low_ctr_visible_page |...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
2,3,77.4,4556,194,16.4,0.33,2.38,stale_visible_content | low_ctr_visible_page |...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
3,4,68.8,7558,193,17.9,0.20,0.00,stale_visible_content | low_ctr_visible_page,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
4,5,65.0,517715,104,4.2,0.14,4.23,low_ctr_visible_page | low_engagement_visible_...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
5,6,65.0,517109,22,5.4,0.25,2.42,low_ctr_visible_page | low_engagement_visible_...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
6,7,65.0,509252,20,2.5,0.15,11.73,low_ctr_visible_page | low_engagement_visible_...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
7,8,64.8,1697,193,15.8,0.12,0.00,stale_visible_content | low_ctr_visible_page,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
8,9,64.7,463103,20,2.3,0.41,11.31,low_ctr_visible_page | low_engagement_visible_...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."
9,10,64.4,416180,22,4.0,0.23,2.37,low_ctr_visible_page | low_engagement_visible_...,review_title_meta_and_intent,"Review after checking intent, seasonality, and..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is for a content editor or SEO lead deciding which pages to inspect first in a limited review cycle. It helps prioritize investigation; it does not predict traffic gain, revenue, page quality, or the result of an edit.

The score is valid only for the data definition and observation window used here. A page can be stale for a good reason, have low CTR because of the search-results layout, or have low engagement because of an expected intent pattern. Therefore, the queue is directional decision support only.


In [3]:
# Review the distribution before treating the queue as a fixed policy.
action_summary = (queue.groupby("suggested_action", as_index=False)
                  .agg(items=("rank", "count"), median_score=("review_score", "median"), median_impressions=("impressions_90d", "median"))
                  .sort_values("items", ascending=False))
print(action_summary.round(1).to_string(index=False))
print("Top-20 review queue: each row needs manual verification before any action.")


            suggested_action  items  median_score  median_impressions
                     monitor  16663          12.8               122.0
review_title_meta_and_intent   9745          43.2              3021.0
review_experience_and_intent   3365          33.5              6965.0
       manual_refresh_review    163          32.0                13.0
    review_depth_and_refresh     64          26.1               426.0
Top-20 review queue: each row needs manual verification before any action.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting, a person must check the current page, search intent, seasonality, indexability, technical issues, product or policy changes, internal-link context, and editorial quality. Reason codes tell the reviewer *why a page entered the queue*; they are not diagnoses.

**No-go list:** do not automatically publish, unpublish, delete, merge, redirect, rewrite, change titles/meta, or allocate budget using this score. Do not reveal client IDs, content IDs, URLs, titles, queries, or raw exports in a public paper. Do not claim that any suggested action will increase visibility.


In [4]:
# A compact human-review checklist accompanies every selected action.
review_checklist = pd.DataFrame({
    "check": ["Confirm current search intent", "Check indexing and technical status", "Check seasonality and recent product/editorial changes", "Read the page and assess content quality", "Document the reviewer decision and rationale"],
    "required_before_action": [True, True, True, True, True],
})
print(review_checklist.to_string(index=False))

                                                 check  required_before_action
                         Confirm current search intent                    True
                   Check indexing and technical status                    True
Check seasonality and recent product/editorial changes                    True
              Read the page and assess content quality                    True
          Document the reviewer decision and rationale                    True


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Re-run the queue when a comparable new measurement window is available. Investigate a retrain or rule review when any of these changes materially:

- the proportion of pages in each action category;
- feature missingness, especially CTR or engagement fields;
- the review-score distribution or the top-50 composition;
- observed Precision@50 from a manually reviewed sample;
- site/client mix, search seasonality, or data-collection definitions.

Any new feature or label definition must repeat the Week-6 leakage audit and grouped validation before use.

In [5]:
# Save monitoring baselines for the next comparable run.
monitoring = pd.DataFrame({
    "metric": ["pages_scored", "top_50_score_threshold", "median_score", "monitor_share", "refresh_review_share"],
    "value": [len(queue), queue.head(50)["review_score"].min(), queue["review_score"].median(), (queue["suggested_action"] == "monitor").mean(), (queue["suggested_action"] != "monitor").mean()],
})
print(monitoring.round(3).to_string(index=False))


                metric     value
          pages_scored 30000.000
top_50_score_threshold    61.700
          median_score    19.600
         monitor_share     0.555
  refresh_review_share     0.445


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The paper should embed only public-safe summaries: a top-20 queue without identifiers, the action-mix table, the monitoring baseline, and the explanation of reason codes and limits. The export below intentionally excludes content IDs, client IDs, URLs, titles, and queries.


In [6]:
public_queue_columns = ["rank", "review_score", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "reason_codes", "suggested_action", "confidence_note"]
queue.loc[:, public_queue_columns].head(100).to_csv(OUT / "w07_ranked_action_playbook.csv", index=False)
action_summary.to_csv(OUT / "w07_action_mix.csv", index=False)
monitoring.to_csv(OUT / "w07_monitoring_baseline.csv", index=False)
print("Wrote public-safe exports to:", OUT)
print("- w07_ranked_action_playbook.csv")
print("- w07_action_mix.csv")
print("- w07_monitoring_baseline.csv")

Wrote public-safe exports to: /content/flyrank_notebook_workspace/work/outputs
- w07_ranked_action_playbook.csv
- w07_action_mix.csv
- w07_monitoring_baseline.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.